# Download do METAR de SBSP (REDEMET)

Coleta das observações METAR do Aeroporto de Congonhas (SBSP) a partir da **API da REDEMET/DECEA**, 
e decodificação do texto bruto com o parser **MetPy**.

Saída: `data/processed/cgh_metar_2025.parquet` — 11.447 observações, 32 colunas,
`date_time` em **UTC**.

## Fonte

- Portal: <https://redemet.decea.mil.br>
- API: <https://api-redemet.decea.mil.br>
- Endpoint: `GET /mensagens/metar/{localidade}?data_ini=&data_fim=&page_tam=&page=`
- Formato de data: `AAAAMMDDHH` (UTC).

A API exige **cadastro gratuito e chave de acesso**. 

In [4]:
# pip install metpy
# pip install pandas

In [2]:
import pandas as pd
import requests
from metpy.io import parse_metar_to_dataframe
from metpy.units import units

## 1. Download das mensagens

A API pagina o resultado. Pedimos a primeira página, lemos `last_page` da
resposta e iteramos até o fim, concatenando tudo num único DataFrame de texto
METAR bruto (coluna `mens`).


In [ ]:
localidades = "SBSP"
data_ini = 2025010101   # 2025-01-01 01Z
data_fim = 2026010103   # 2026-01-01 03Z
page_tam = 150

BASE = "https://api-redemet.decea.mil.br/mensagens/metar"
API_KEY = "XXX"

def busca_pagina(page=None):
    url = (
        f"{BASE}/{localidades}?api_key={API_KEY}"
        f"&data_ini={data_ini}&data_fim={data_fim}&page_tam={page_tam}"
    )
    if page is not None:
        url += f"&page={page}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()


df = busca_pagina()
metar = pd.json_normalize(df["data"]["data"])
last_page = pd.json_normalize(df["data"])["last_page"].values[0]

for page in range(2, last_page + 1):
    print(f"Página {page} de {last_page}")
    df = busca_pagina(page)
    metar = pd.concat(
        [metar, pd.json_normalize(df["data"]["data"])], ignore_index=True
    )

print(f"{len(metar)} mensagens METAR baixadas")
metar.head()


## 2. Decodificação com MetPy

Cada mensagem é decodificada individualmente com `parse_metar_to_dataframe`. O
METAR não carrega ano nem mês (só dia/hora), então ambos vêm de
`validade_inicial`, devolvido pela REDEMET.

Mensagens malformadas são reportadas e descartadas — em 2025 a taxa foi
desprezível, mas a contagem final deve ser conferida contra o total baixado.

`altimeter_hpa` é derivada aqui: o MetPy devolve o altímetro em polegadas de
mercúrio, e a conversão para hPa deixa o valor no formato usado na aviação.


In [ ]:
decoded_rows = []
for _, row in metar.iterrows():
    metar_str = row["mens"]
    date = pd.to_datetime(row["validade_inicial"])
    try:
        decoded = parse_metar_to_dataframe(metar_str, year=date.year, month=date.month)
        decoded_rows.append(decoded)
    except Exception as e:
        print(f"Falha ao decodificar: {metar_str[:30]}... ({e})")

decoded_df = pd.concat(decoded_rows, ignore_index=True)
decoded_df["altimeter_hpa"] = (
    (decoded_df["altimeter"].values * units.inHg).to("hPa").magnitude.round(0)
)

print(f"{len(decoded_df)} de {len(metar)} mensagens decodificadas")
decoded_df.head()


## 3. Gravação

O arquivo resultante alimenta o notebook `09_merge_metar_modelo`, onde o METAR
é pareado com cada decolagem (observação válida mais recente antes do horário
de decolagem) e convertido nas features de visibilidade, teto e fenômenos
significativos.


In [ ]:
decoded_df.to_parquet("data/processed/cgh_metar_2025.parquet", index=False)